# Multi-Agent AI Systems Workshop - STARTER
## Build It Yourself!

**2 Project Tracks:**
- **Track A**: Document Intelligence Hub (6 agents, parallel pipeline)
- **Track B**: Multi-Agent Debate System (7 agents, adversarial debate)

Both tracks use **Groq API** (free tier, no credit card needed) with **LLaMA models**.
Embeddings run locally via `sentence-transformers` (no API limits).

**What's given:** Agent functions (they work individually).
**Your job:** Build the ENGINEERING that makes them work together.

---

## 1. Setup - Install Dependencies & Set API Key

Get a **free** Groq API key at: https://console.groq.com/keys (sign up with email or Google, no credit card needed)

In [ ]:
# Install required packages
!pip install -q groq sentence-transformers pymupdf faiss-cpu pydantic python-dotenv numpy

In [ ]:
# Set your Groq API key
# Get a free key at: https://console.groq.com/keys
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in left sidebar -> Add secret named GROQ_API_KEY
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('API key loaded from Colab Secrets')
except:
    # Option 2: Paste directly (less secure)
    os.environ['GROQ_API_KEY'] = 'YOUR_KEY_HERE'  # <-- Replace with your key
    print('API key set manually')

## 2. helpers.py - Shared Building Blocks (GIVEN)

Everything you need is here:
- `load_and_chunk(pdf_path)` - PDF to text chunks
- `build_index(chunks)` - FAISS vector index (embeddings via local sentence-transformers)
- `search(index, chunks, query)` - Semantic search
- `call_llm_cheap()` / `call_llm_strong()` - LLM calls via Groq with retry
- `parse_json(text)` - Safe JSON extraction
- `CostTracker` - Token/cost tracking
- `EvalHarness` - Pipeline benchmarking

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*unauthenticated.*HF Hub.*")
import logging
logging.getLogger().setLevel(logging.ERROR)
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import fitz  # PyMuPDF
import faiss
import numpy as np
import json
import time
import re
import hashlib
from groq import Groq
from sentence_transformers import SentenceTransformer

# Configure Groq client
_client = Groq(api_key=os.environ['GROQ_API_KEY'])
PROVIDER = 'groq'
print('[helpers] Using Groq API (LLaMA models)')

# Load local embedding model (runs on CPU, no API needed)
print('[helpers] Loading embedding model (one-time download ~80MB)...')
_embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('[helpers] Embedding model ready (384 dimensions)')

# ============================================================
# COST TRACKING
# ============================================================

MODEL_PRICING = {
    'llama-3.1-8b-instant':     {'input': 0.05,  'output': 0.08},
    'llama-3.3-70b-versatile':  {'input': 0.59,  'output': 0.79},
    'local-embedding':          {'input': 0.00,  'output': 0.00},
}


class CostTracker:
    """Tracks token usage and estimated cost across all LLM calls."""

    def __init__(self, budget=1.00):
        self.budget = budget
        self.calls = []
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0.0

    def record(self, llm_result, agent_name='unknown'):
        tokens = llm_result.get('tokens', {})
        model = llm_result.get('model', 'llama-3.1-8b-instant')
        inp = tokens.get('input', 0)
        out = tokens.get('output', 0)
        pricing = MODEL_PRICING.get(model, {'input': 1.0, 'output': 3.0})
        cost = (inp * pricing['input'] + out * pricing['output']) / 1_000_000
        self.calls.append({'agent': agent_name, 'model': model,
                           'input_tokens': inp, 'output_tokens': out,
                           'cost': cost, 'latency_ms': llm_result.get('latency_ms', 0)})
        self.total_input_tokens += inp
        self.total_output_tokens += out
        self.total_cost += cost
        return cost

    def remaining(self):
        return max(0, self.budget - self.total_cost)

    def check_budget(self):
        if self.total_cost > self.budget:
            raise RuntimeError(f'Budget exceeded: ${self.total_cost:.4f} of ${self.budget:.2f}')

    def report(self):
        print('\n' + '=' * 65)
        print('  COST REPORT')
        print('=' * 65)
        agent_costs = {}
        for call in self.calls:
            name = call['agent']
            if name not in agent_costs:
                agent_costs[name] = {'calls': 0, 'tokens': 0, 'cost': 0, 'latency': 0}
            agent_costs[name]['calls'] += 1
            agent_costs[name]['tokens'] += call['input_tokens'] + call['output_tokens']
            agent_costs[name]['cost'] += call['cost']
            agent_costs[name]['latency'] += call['latency_ms']
        print(f"\n  {'Agent':<22} {'Calls':>5} {'Tokens':>8} {'Cost':>9} {'Latency':>9}")
        print(f"  {'-' * 57}")
        for name, data in sorted(agent_costs.items(), key=lambda x: -x[1]['cost']):
            print(f"  {name:<22} {data['calls']:>5} {data['tokens']:>8} ${data['cost']:>7.4f} {data['latency']:>7}ms")
        print(f"  {'-' * 57}")
        print(f"  {'TOTAL':<22} {len(self.calls):>5} "
              f"{self.total_input_tokens + self.total_output_tokens:>8} "
              f"${self.total_cost:>7.4f} "
              f"{sum(c['latency_ms'] for c in self.calls):>7}ms")
        print(f"\n  Budget: ${self.budget:.2f} | Spent: ${self.total_cost:.4f} | "
              f"Remaining: ${self.remaining():.4f}")
        print('=' * 65)

    def to_dict(self):
        return {'total_calls': len(self.calls), 'total_input_tokens': self.total_input_tokens,
                'total_output_tokens': self.total_output_tokens,
                'total_cost_usd': round(self.total_cost, 6),
                'budget_usd': self.budget, 'remaining_usd': round(self.remaining(), 6),
                'calls': self.calls}


# ============================================================
# DOCUMENT PROCESSING
# ============================================================

def load_and_chunk(pdf_path, chunk_size=300, overlap=50):
    """Load a PDF and split into overlapping text chunks."""
    doc = fitz.open(pdf_path)
    pages = len(doc)
    text = ' '.join([page.get_text() for page in doc])
    doc.close()
    words = text.split()
    if not words:
        raise ValueError(f'No text found in {pdf_path}.')
    chunks = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    print(f'  Loaded {pages} pages, {len(words)} words -> {len(chunks)} chunks '
          f'(size={chunk_size}, overlap={overlap})')
    return chunks


# ============================================================
# EMBEDDINGS (Local - no API needed, no rate limits!)
# ============================================================

def embed(text):
    """Convert text into a vector embedding using local sentence-transformers."""
    return _embed_model.encode(text, convert_to_numpy=True).astype('float32')


def build_index(chunks):
    """Build a FAISS index from text chunks."""
    print(f'  Embedding {len(chunks)} chunks (local model, no API limits)...')
    vecs = _embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True).astype('float32')
    index = faiss.IndexFlatL2(vecs.shape[1])
    index.add(vecs)
    print(f'  Index ready: {index.ntotal} vectors, {vecs.shape[1]} dimensions')
    return index


def search(index, chunks, query, k=5):
    """Search FAISS index for the k most similar chunks to query."""
    query_vec = embed(query).reshape(1, -1)
    distances, indices = index.search(query_vec, min(k, index.ntotal))
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(chunks):
            results.append({'text': chunks[idx], 'score': round(float(1 / (1 + dist)), 4),
                            'index': int(idx)})
    return results


# ============================================================
# LLM CALLING (Groq - blazing fast inference)
# ============================================================

def call_llm(prompt, system='You are a helpful assistant.',
             model=None, temperature=0, max_tokens=2000, json_output=False,
             retries=2, backoff_base=2.0):
    """Call the Groq LLM with automatic retry on transient failures."""
    last_error = None
    for attempt in range(retries + 1):
        try:
            start = time.time()
            m = model or 'llama-3.1-8b-instant'

            messages = [
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': prompt}
            ]
            if json_output:
                messages[1]['content'] += '\n\nIMPORTANT: Return ONLY valid JSON. No explanation, no markdown code fences.'

            kwargs = {
                'model': m,
                'messages': messages,
                'temperature': temperature,
                'max_tokens': max_tokens,
            }
            if json_output:
                kwargs['response_format'] = {'type': 'json_object'}

            response = _client.chat.completions.create(**kwargs)
            elapsed = time.time() - start

            usage = response.usage
            input_tokens = usage.prompt_tokens if usage else 0
            output_tokens = usage.completion_tokens if usage else 0

            return {
                'text': response.choices[0].message.content,
                'tokens': {'input': input_tokens, 'output': output_tokens},
                'latency_ms': int(elapsed * 1000),
                'model': m
            }
        except Exception as e:
            last_error = e
            if attempt < retries:
                wait = backoff_base ** attempt
                print(f'    [retry] Attempt {attempt + 1} failed: {e}. Waiting {wait:.0f}s...')
                time.sleep(wait)
            else:
                raise last_error


def call_llm_cheap(prompt, system='You are a helpful assistant.',
                   temperature=0, max_tokens=1000, json_output=False):
    """Cheap/fast model: LLaMA 3.1 8B. Use for planners and simple routing."""
    return call_llm(prompt, system, model='llama-3.1-8b-instant',
                    temperature=temperature, max_tokens=max_tokens, json_output=json_output)


def call_llm_strong(prompt, system='You are a helpful assistant.',
                    temperature=0, max_tokens=8192, json_output=False):
    """Strong model: LLaMA 3.3 70B. Use for reasoning, debate, judgment."""
    return call_llm(prompt, system, model='llama-3.3-70b-versatile',
                    temperature=temperature, max_tokens=max_tokens, json_output=json_output)


# ============================================================
# JSON PARSING
# ============================================================

def parse_json(text):
    """Safely parse JSON from LLM output. Handles arrays and markdown fences."""
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```$', '', text.strip())

    def _wrap_if_list(parsed):
        if not isinstance(parsed, list):
            return parsed
        if not parsed or not isinstance(parsed[0], dict):
            return {'items': parsed}
        first = parsed[0]
        if 'fact' in first:
            return {'facts': parsed, 'total_extracted': len(parsed)}
        if 'question' in first:
            return {'questions': parsed}
        if 'topic' in first and 'severity' in first:
            return {'gaps': parsed, 'coverage_score': 0.75, 'recommendation': 'See gaps above.'}
        if 'score' in first or 'overall_score' in first:
            return parsed[0] if len(parsed) == 1 else {'items': parsed}
        return {'items': parsed}

    try:
        return _wrap_if_list(json.loads(text))
    except json.JSONDecodeError:
        pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return _wrap_if_list(json.loads(match.group()))
        except json.JSONDecodeError:
            pass
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        try:
            return _wrap_if_list(json.loads(match.group()))
        except json.JSONDecodeError:
            pass
    raise ValueError(f'Could not parse JSON from LLM output:\n{text[:300]}...')


# ============================================================
# SEMANTIC CACHE
# ============================================================

class SemanticCache:
    """Cache LLM responses by query similarity."""
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.entries = []

    def _cosine_sim(self, a, b):
        dot = np.dot(a, b)
        norm = np.linalg.norm(a) * np.linalg.norm(b)
        return float(dot / norm) if norm > 0 else 0.0

    def get(self, query_text, query_vec=None):
        if not self.entries:
            return None
        if query_vec is None:
            query_vec = embed(query_text)
        for _, cached_vec, cached_response in self.entries:
            if self._cosine_sim(query_vec, cached_vec) >= self.threshold:
                return cached_response
        return None

    def put(self, query_text, response, query_vec=None):
        if query_vec is None:
            query_vec = embed(query_text)
        self.entries.append((query_text, query_vec, response))


# ============================================================
# EVALUATION HARNESS
# ============================================================

class EvalHarness:
    """Benchmark your pipeline against test cases."""
    def __init__(self):
        self.test_cases = []
        self.results = []

    def add_test(self, question, expected_keywords=None, expected_answer=None, difficulty='medium'):
        self.test_cases.append({'question': question, 'expected_keywords': expected_keywords or [],
                                'expected_answer': expected_answer, 'difficulty': difficulty})

    def run(self, pipeline_fn):
        self.results = []
        print(f'\n  Running {len(self.test_cases)} evaluation tests...')
        for i, test in enumerate(self.test_cases, 1):
            start = time.time()
            try:
                result = pipeline_fn(test['question'])
                elapsed = time.time() - start
                answer = result.get('answer', result.get('report', ''))
                critic_score = result.get('critic_score', 0)
                answer_lower = answer.lower() if isinstance(answer, str) else str(answer).lower()
                keywords_found = sum(1 for kw in test['expected_keywords'] if kw.lower() in answer_lower)
                keyword_score = keywords_found / len(test['expected_keywords']) if test['expected_keywords'] else 1.0
                composite = keyword_score * 0.5 + critic_score * 0.5
                self.results.append({'question': test['question'], 'difficulty': test['difficulty'],
                                     'keyword_score': round(keyword_score, 2), 'critic_score': round(critic_score, 2),
                                     'composite_score': round(composite, 2), 'latency_s': round(elapsed, 1),
                                     'status': 'pass' if composite >= 0.6 else 'fail'})
            except Exception as e:
                self.results.append({'question': test['question'], 'status': 'error', 'error': str(e),
                                     'composite_score': 0, 'latency_s': round(time.time() - start, 1)})
            s = self.results[-1]
            icon = 'PASS' if s['status'] == 'pass' else 'FAIL' if s['status'] == 'fail' else 'ERR'
            print(f"  {i}. [{icon}] {test['question'][:50]}... (score: {s['composite_score']})")
        return self.results

    def report(self):
        if not self.results: return
        passed = sum(1 for r in self.results if r['status'] == 'pass')
        total = len(self.results)
        avg = sum(r['composite_score'] for r in self.results) / total
        print(f'\n  Results: {passed}/{total} passed ({passed/total*100:.0f}%), Avg score: {avg:.2f}')


# ============================================================
# STATE MANAGEMENT
# ============================================================

def init_state(query=''):
    return {'query': query, 'chunks': [], 'log': [], 'errors': [], 'start_time': time.time()}

def log_agent(state, agent_name, input_summary, output_summary, meta=None):
    entry = {'agent': agent_name, 'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ'),
             'input': input_summary, 'output': output_summary}
    if meta: entry.update(meta)
    state['log'].append(entry)
    return state

def print_log(state):
    print('\n' + '=' * 65)
    print('  AGENT EXECUTION LOG')
    print('=' * 65)
    for entry in state['log']:
        print(f"\n  [{entry['agent'].upper()}] @ {entry.get('timestamp', '?')}")
        if 'tokens' in entry:
            t = entry['tokens']
            print(f"    Tokens: {t.get('input', 0)} in / {t.get('output', 0)} out")
        if 'latency_ms' in entry:
            print(f"    Latency: {entry['latency_ms']}ms")
        output = entry.get('output', '')
        if isinstance(output, str):
            print(f"    Output: {output[:120]}")
    elapsed = time.time() - state.get('start_time', time.time())
    print(f"\n  Total time: {elapsed:.1f}s | Agents called: {len(state['log'])}")
    print('=' * 65)

print('helpers loaded successfully!')

## 3. Test helpers - Verify Everything Works

In [ ]:
print('1. Testing LLM call...')
result = call_llm_cheap("Say 'Hello Workshop!' and nothing else.")
print(f'   Response: {result["text"]}')

print('\n2. Testing CostTracker...')
tracker = CostTracker(budget=0.50)
tracker.record(result, agent_name='test')
tracker.report()

print('\n3. Testing JSON output + parse...')
result = call_llm_cheap('Return JSON: {"status": "ok"}', json_output=True)
parsed = parse_json(result['text'])
print(f'   Parsed: {parsed}')

print('\n4. Testing embedding...')
vec = embed('machine learning is great')
print(f'   Vector shape: {vec.shape}')

print('\n5. Testing SemanticCache...')
cache = SemanticCache(threshold=0.95)
cache.put('What is ML?', {'answer': 'Machine learning is...'})
hit = cache.get('What is machine learning?')
print(f'   Cache hit: {hit is not None}')

print('\nAll tests passed! Ready for labs.')

## 4. Create a Test PDF

Upload your own PDF, or use this auto-generated test document:

In [ ]:
# Create a test PDF document
doc = fitz.open()
page = doc.new_page()
text = """Machine Learning: A Comprehensive Overview

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:

1. Supervised Learning: The algorithm learns from labeled training data, making predictions or
decisions based on past examples. Common algorithms include Linear Regression, Decision Trees,
Random Forests, and Support Vector Machines.

2. Unsupervised Learning: The algorithm finds patterns in unlabeled data without predefined
categories. K-Means Clustering, PCA, and DBSCAN are popular unsupervised methods.

3. Reinforcement Learning: An agent learns to make decisions by performing actions and receiving
rewards or penalties. Used in robotics, game playing, and autonomous vehicles.

Key Concepts:
- Feature Engineering: Selecting and transforming variables for model input.
- Overfitting: When a model learns noise instead of the underlying pattern.
- Cross-Validation: A technique for assessing model performance on unseen data.
- Bias-Variance Tradeoff: Balancing model complexity and generalization.
- Gradient Descent: An optimization algorithm used to minimize the loss function.

Applications:
Machine learning powers recommendation systems (Netflix, Spotify), natural language processing
(ChatGPT, Google Translate), computer vision (autonomous driving, medical imaging), fraud
detection in banking, and predictive maintenance in manufacturing.

The field continues to evolve rapidly with deep learning, transformer architectures, and
large language models pushing the boundaries of what AI can achieve.

Evaluation Metrics:
- Accuracy: Percentage of correct predictions.
- Precision: Of all positive predictions, how many were actually positive.
- Recall: Of all actual positives, how many were correctly identified.
- F1 Score: Harmonic mean of precision and recall.
- AUC-ROC: Area under the receiver operating characteristic curve.

Challenges:
Data quality, interpretability, computational costs, ethical considerations, and the need for
domain expertise remain significant challenges in practical ML deployments.
"""
page.insert_text((72, 72), text, fontsize=10)
doc.save('test_doc.pdf')
doc.close()
print('test_doc.pdf created!')

# Or upload your own PDF:
# from google.colab import files
# uploaded = files.upload()  # Upload dialog
# PDF_PATH = list(uploaded.keys())[0]

PDF_PATH = 'test_doc.pdf'

---
# Track A: Document Intelligence Hub

**Architecture:**
```
Planner -> [Summarizer | Fact Extractor | Quiz Gen | Gap Analyzer] -> Critic -> Report
                                    ^                                    |
                                    +-- adaptive retry (critic feedback) +
```

**5 Milestones:**
1. Wire sequential pipeline
2. Parallel execution with ThreadPoolExecutor
3. Cost tracking + budget enforcement
4. Evaluation harness
5. Adaptive retry (Critic feedback loops to Planner)

## Track A - Agent Functions (GIVEN)

These agent functions are complete and work individually. **Don't modify them.**
Your job is to wire them together into a pipeline in the milestones below.

In [ ]:
# ============================================================
# AGENT PROMPTS (GIVEN)
# ============================================================

PLANNER_SYSTEM = """You are a Document Planner agent.
Analyze the document and identify structure, themes, and complexity.
Return JSON:
{
    "document_type": "textbook chapter | research paper | policy doc | report | other",
    "main_topic": "one clear sentence",
    "key_themes": ["theme1", "theme2", "theme3", "theme4", "theme5"],
    "target_audience": "who this is written for",
    "estimated_complexity": "beginner | intermediate | advanced"
}"""

SUMMARIZER_SYSTEM = """You are a Summarizer agent. Write a concise executive summary.
Rules: Max 5 sentences. Every statement from chunks only. No training data.
Return JSON: {"summary": "3-5 sentences", "key_points": ["point1", "point2", "point3"]}"""

FACT_EXTRACTOR_SYSTEM = """You are a Fact Extractor agent. Extract 6-10 key facts.
Classify each as: fact | claim | formula | statistic. Rate: high | medium | low.
Return JSON: {"facts": [{"fact": "...", "type": "...", "importance": "..."}], "total_extracted": N}"""

QUIZ_SYSTEM = """You are a Quiz Generator agent. Create 5 MCQs testing UNDERSTANDING, not recall.
4 options (A-D), 1 correct. Mix: 2 easy, 2 medium, 1 hard. Include explanation.
Return JSON: {"questions": [{"question": "...", "options": {"A":"..","B":"..","C":"..","D":".."}, "correct": "B", "explanation": "...", "difficulty": "easy|medium|hard"}]}"""

GAP_ANALYZER_SYSTEM = """You are a Gap Analyzer agent. Find 3-5 missing or underexplained topics.
Be specific. Rate: critical | moderate | minor.
Return JSON: {"gaps": [{"topic": "...", "why_important": "...", "severity": "..."}], "coverage_score": 0.75, "recommendation": "one sentence"}"""

CRITIC_SYSTEM = """You are a Critic agent (quality gate). Verify all outputs against source chunks.
Be STRICT. Score 0 for any unsupported claim.
Return JSON:
{"overall_score": 0.85, "scores": {"summary": {"score": 0.9, "issues": []}, "facts": {"score": 0.8, "issues": []}, "quiz": {"score": 0.85, "issues": []}, "gaps": {"score": 0.9, "issues": []}}, "verdict": "pass|fail", "critical_issues": [], "improvement_hints": ["what to fix on retry"]}"""


# ============================================================
# AGENT FUNCTIONS (GIVEN - these work individually)
# ============================================================

def planner(state, tracker):
    sample_text = '\n---\n'.join([c['text'] for c in state['chunks'][:8]])
    retry_context = ''
    if state.get('_critic_feedback'):
        retry_context = f"\n\nPrevious attempt issues: {state['_critic_feedback']}\nFocus the themes to address these gaps."
    result = call_llm_cheap(system=PLANNER_SYSTEM,
                            prompt=f'Analyze this document:\n\n{sample_text}{retry_context}',
                            json_output=True)
    tracker.record(result, 'planner')
    parsed = parse_json(result['text'])
    state['plan'] = parsed
    log_agent(state, 'planner', {'chunks': 8}, parsed,
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Planner] Topic: {parsed.get('main_topic', '?')}")
    return state


def summarizer(state, tracker):
    chunks_text = '\n---\n'.join([c['text'] for c in state['chunks'][:15]])
    plan = state.get('plan', {})
    result = call_llm_strong(
        system=SUMMARIZER_SYSTEM,
        prompt=f"Topic: {plan.get('main_topic', '?')}\nThemes: {', '.join(plan.get('key_themes', []))}\n\nChunks:\n{chunks_text}",
        json_output=True)
    tracker.record(result, 'summarizer')
    state['summary'] = parse_json(result['text'])
    log_agent(state, 'summarizer', {'chunks': 15}, state['summary'].get('summary', '')[:80],
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Summarizer] Done")
    return state


def fact_extractor(state, tracker):
    chunks_text = '\n---\n'.join([c['text'] for c in state['chunks'][:15]])
    result = call_llm_strong(system=FACT_EXTRACTOR_SYSTEM,
                             prompt=f'Extract key facts:\n\n{chunks_text}', json_output=True)
    tracker.record(result, 'fact_extractor')
    state['facts'] = parse_json(result['text'])
    log_agent(state, 'fact_extractor', {'chunks': 15},
              f"{len(state['facts'].get('facts', []))} facts",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Fact Extractor] {len(state['facts'].get('facts', []))} facts")
    return state


def quiz_generator(state, tracker):
    chunks_text = '\n---\n'.join([c['text'] for c in state['chunks'][:15]])
    plan = state.get('plan', {})
    result = call_llm_strong(
        system=QUIZ_SYSTEM,
        prompt=f"Topic: {plan.get('main_topic', '?')}\nSource:\n{chunks_text}",
        json_output=True)
    tracker.record(result, 'quiz_generator')
    state['quiz'] = parse_json(result['text'])
    log_agent(state, 'quiz_generator', {'chunks': 15},
              f"{len(state['quiz'].get('questions', []))} questions",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Quiz Gen] {len(state['quiz'].get('questions', []))} questions")
    return state


def gap_analyzer(state, tracker):
    chunks_text = '\n---\n'.join([c['text'] for c in state['chunks'][:15]])
    plan = state.get('plan', {})
    result = call_llm_strong(
        system=GAP_ANALYZER_SYSTEM,
        prompt=f"Type: {plan.get('document_type', '?')}\nTopic: {plan.get('main_topic', '?')}\n\n{chunks_text}",
        json_output=True)
    tracker.record(result, 'gap_analyzer')
    state['gaps'] = parse_json(result['text'])
    log_agent(state, 'gap_analyzer', {'chunks': 15},
              f"{len(state['gaps'].get('gaps', []))} gaps",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Gap Analyzer] {len(state['gaps'].get('gaps', []))} gaps")
    return state


def critic(state, tracker):
    chunks_text = '\n---\n'.join([c['text'] for c in state['chunks'][:10]])
    outputs = {
        'summary': state.get('summary', {}).get('summary', 'MISSING'),
        'facts_sample': [f['fact'] for f in state.get('facts', {}).get('facts', [])[:3]],
        'quiz_sample': state.get('quiz', {}).get('questions', [{}])[0].get('question', 'MISSING'),
        'gaps_count': len(state.get('gaps', {}).get('gaps', []))
    }
    result = call_llm_strong(
        system=CRITIC_SYSTEM,
        prompt=f'Source chunks:\n{chunks_text}\n\nOutputs to validate:\n{json.dumps(outputs, indent=2)}',
        json_output=True)
    tracker.record(result, 'critic')
    parsed = parse_json(result['text'])
    state['critic'] = parsed
    state['critic_score'] = parsed.get('overall_score', 0)
    state['_critic_feedback'] = '; '.join(parsed.get('improvement_hints', parsed.get('critical_issues', [])))
    log_agent(state, 'critic', {'outputs': 4},
              f"Score={state['critic_score']}, Verdict={parsed.get('verdict', '?')}",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Critic] Score: {state['critic_score']}/1.0 - {parsed.get('verdict', '?').upper()}")
    return state

print('Track A agents loaded!')

## Milestone 1: Wire the Sequential Pipeline

**Your task:** Call agents in the correct order with proper state flow.

**Pipeline:** Load PDF -> Build Index -> Select Chunks -> Planner -> Analysis Agents -> Critic -> Report

**Hints:**
- `chunks = load_and_chunk(pdf_path)` returns raw text chunks
- `index = build_index(chunks)` creates FAISS index
- `search(index, chunks, query, k=5)` returns `[{text, score, index}]`
- Search for different aspects ("overview", "methodology", "conclusion") and deduplicate by index
- Store selected chunks in `state["chunks"]`
- Run agents sequentially for now: `planner -> summarizer -> fact_extractor -> quiz_generator -> gap_analyzer -> critic`

In [ ]:
def run_hub_pipeline(pdf_path, budget=1.00, max_retries=1):
    """Run the full Document Intelligence Hub pipeline."""
    tracker = CostTracker(budget=budget)
    print(f"\n{'=' * 65}\n  DOCUMENT INTELLIGENCE HUB\n{'=' * 65}")
    print(f'  Document: {pdf_path}\n  Budget: ${budget:.2f}')

    # Step 1: Load + index
    print('\n[1/6] Loading and indexing...')
    # YOUR CODE HERE
    # HINT: chunks = load_and_chunk(pdf_path)
    # HINT: index = build_index(chunks)


    # Step 2: Select diverse chunks via semantic search
    print('\n[2/6] Selecting diverse chunks...')
    state = init_state()
    # YOUR CODE HERE
    # HINT: Search for different aspects to get diversity:
    #   queries = ['main topic overview', 'key concept definition',
    #              'important conclusion', 'methodology approach', 'example illustration']
    #   for q in queries: all_results.extend(search(index, chunks, q, k=4))
    # HINT: Deduplicate by index: seen = set(); [c for c in results if c['index'] not in seen and not seen.add(c['index'])]
    # HINT: Store in state['chunks'] (take up to 20)


    # Step 3: Planner
    print('\n[3/6] Planner...')
    # YOUR CODE HERE
    # HINT: state = planner(state, tracker)


    # Step 4: Run 4 analysis agents SEQUENTIALLY
    print('\n[4/6] Analysis agents...')
    # YOUR CODE HERE
    # HINT: Call each agent in order:
    #   state = summarizer(state, tracker)
    #   state = fact_extractor(state, tracker)
    #   state = quiz_generator(state, tracker)
    #   state = gap_analyzer(state, tracker)


    # Step 5: Critic (quality gate)
    print('\n[5/6] Critic...')
    # YOUR CODE HERE
    # HINT: state = critic(state, tracker)


    # Step 6: Print results
    print('\n[6/6] Results:')
    # YOUR CODE HERE
    # HINT: Print critic_score, summary, fact count, question count
    # print(f"  Quality: {state['critic_score']}/1.0")
    # print(f"  Summary: {state['summary']['summary'][:200]}")
    # print(f"  Facts: {len(state['facts']['facts'])}")
    # print(f"  Questions: {len(state['quiz']['questions'])}")


    tracker.report()
    return state

# Uncomment when ready:
# hub_state = run_hub_pipeline(PDF_PATH)

## Milestone 2: Parallel Execution

**Your task:** Run the 4 analysis agents in parallel using `ThreadPoolExecutor`.

**Requirements:**
1. Each agent gets its OWN copy of state: `agent_state = {**state}`
2. Wrap `future.result()` in try/except (one failure shouldn't crash all)
3. Merge results back: `state['summary'] = result_state['summary']`
4. Merge logs: `state['log'].extend(result_state.get('log', []))`

**Test:** Time before vs after - should be ~3x faster.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import traceback

def run_parallel_agents(state, tracker):
    """Run 4 analysis agents in parallel with error boundaries."""
    print('\n  Running 4 analysis agents in parallel...')

    # YOUR CODE HERE
    # HINT: Define the agents and their state keys:
    #   agents = {'summarizer': summarizer, 'fact_extractor': fact_extractor,
    #             'quiz_generator': quiz_generator, 'gap_analyzer': gap_analyzer}
    #   key_map = {'summarizer': 'summary', 'fact_extractor': 'facts',
    #              'quiz_generator': 'quiz', 'gap_analyzer': 'gaps'}
    #
    # HINT: Submit to ThreadPoolExecutor:
    #   with ThreadPoolExecutor(max_workers=4) as executor:
    #       futures = {}
    #       for name, fn in agents.items():
    #           agent_state = {**state}  # Each agent gets its own copy
    #           futures[executor.submit(fn, agent_state, tracker)] = name
    #       for future in as_completed(futures):
    #           name = futures[future]
    #           try:
    #               result_state = future.result()
    #               state[key_map[name]] = result_state[key_map[name]]
    #               state['log'].extend(result_state.get('log', []))
    #           except Exception as e:
    #               print(f'  [ERROR] {name} failed: {e}')

    return state

# Update run_hub_pipeline Step 4 to use run_parallel_agents(state, tracker) instead of sequential calls

## Milestone 3: Cost Tracking + Budget Enforcement

**Your task:** Add `tracker.check_budget()` between pipeline stages.

**Questions to answer:**
- Where should you place budget checks? After every agent? After each stage? Only at the end?
- What happens if you run with budget=$0.05?

**Test:** Try calling `run_hub_pipeline(PDF_PATH, budget=0.05)` and see if it stops gracefully.

In [ ]:
# YOUR CODE HERE
# Go back to run_hub_pipeline and add tracker.check_budget() calls
# between major stages (after planner, after analysis, etc.)
# Wrap in try/except RuntimeError for graceful budget exceeded message

# Test with tight budget:
# hub_state = run_hub_pipeline(PDF_PATH, budget=0.05)

## Milestone 4: Adaptive Retry

**Your task:** When Critic scores < 0.7, feed feedback back to Planner and retry.

**Requirements:**
1. Build a retry loop: `while retry < max_retries and score < 0.7`
2. `state['_critic_feedback']` already contains Critic's hints (Planner reads it automatically)
3. Re-run: planner (with feedback) -> parallel agents -> critic
4. Budget check before each retry

**Test:** Does the score improve on retry? Check the log to see if Planner adapted.

In [ ]:
# YOUR CODE HERE
# Modify run_hub_pipeline to add adaptive retry around steps 4-5
#
# HINT:
# retry_count = 0
# while retry_count <= max_retries:
#     tracker.check_budget()
#     state = run_parallel_agents(state, tracker)  # Step 4
#     state = critic(state, tracker)                # Step 5
#     if state.get('critic_score', 0) >= 0.7:
#         print(f'  PASSED (score: {state["critic_score"]})')
#         break
#     else:
#         print(f'  FAILED (score: {state["critic_score"]}) - retrying...')
#         if retry_count < max_retries:
#             state = planner(state, tracker)  # Planner reads _critic_feedback
#         retry_count += 1

# Test:
# hub_state = run_hub_pipeline(PDF_PATH, budget=1.00, max_retries=2)

---
# Track B: Multi-Agent Debate System

**Architecture:**
```
Planner -> [Researcher FOR | Researcher AGAINST] -> [Debater FOR | Debater AGAINST]
        -> Judge R1 -> (if close) [Cross-Exam FOR | AGAINST] -> Judge R2 -> Synthesizer
```

**5 Milestones:**
1. Wire sequential debate pipeline
2. Parallel researchers + debaters
3. Cross-examination round
4. Cost tracking + budget decisions
5. Tournament mode (3 topics)

## Track B - Agent Functions (GIVEN)

These agent functions are complete. **Don't modify them.**
Your job is to wire them into a debate pipeline.

In [ ]:
# ============================================================
# DEBATE AGENT PROMPTS (GIVEN)
# ============================================================

DEBATE_PLANNER_SYSTEM = """You are a Debate Planner. Set up a fair debate framework.
Return JSON:
{"topic_restated": "...", "for_position": "FOR argues (1 sentence)", "against_position": "AGAINST argues (1 sentence)", "dimensions": ["dim1", "dim2", "dim3"], "context_from_document": "..."}"""

DEBATER_SYSTEM_TEMPLATE = """You are arguing {side} the position: \"{position}\"
Rules: Use ONLY evidence from chunks. 3 arguments with point/evidence/reasoning. Preemptive rebuttal.
Return JSON:
{{"opening_statement": "...", "arguments": [{{"point": "...", "evidence": "...", "reasoning": "..."}}], "counter_to_opposition": "...", "closing_statement": "..."}}"""

CROSS_EXAM_SYS = """You are a Cross-Examiner challenging the opposing argument.
Find the weakest point. Cite evidence that contradicts it.
Return JSON:
{"weakest_point": "...", "challenge": "...", "unanswerable_questions": ["q1", "q2"], "additional_evidence": "..."}"""

JUDGE_SYS = """You are an impartial Judge. Score both sides on 5 criteria (0-10 each):
Evidence Quality, Logical Coherence, Completeness, Persuasiveness, Honesty.
Return JSON:
{"for_score": {"evidence_quality": 8, "logical_coherence": 7, "completeness": 8, "persuasiveness": 7, "honesty": 9, "total": 39}, "against_score": {"evidence_quality": 7, "logical_coherence": 8, "completeness": 7, "persuasiveness": 8, "honesty": 8, "total": 38}, "winner": "for|against|tie", "margin": "decisive|narrow|razor-thin", "reasoning": "2-3 sentences", "strongest_point_for": "...", "strongest_point_against": "...", "weakest_point_for": "...", "weakest_point_against": "..."}"""

SYNTHESIZER_SYS = """You are a Synthesizer. Balanced analysis beyond 'both sides have merit.'
150-200 words. Common ground + key tension + nuanced conclusion.
Return JSON: {"balanced_analysis": "...", "common_ground": ["..."], "key_tension": "...", "nuanced_conclusion": "..."}"""


# ============================================================
# DEBATE AGENT FUNCTIONS (GIVEN)
# ============================================================

def debate_planner(state, tracker):
    sample_text = '\n---\n'.join([c['text'] for c in state['chunks'][:6]])
    result = call_llm_cheap(system=DEBATE_PLANNER_SYSTEM,
                            prompt=f"Debate topic: {state['topic']}\n\nDocument:\n{sample_text}",
                            json_output=True)
    tracker.record(result, 'planner')
    state['debate_plan'] = parse_json(result['text'])
    log_agent(state, 'planner', {'topic': state['topic']}, state['debate_plan'],
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Planner] FOR: {state['debate_plan'].get('for_position', '?')[:60]}")
    print(f"  [Planner] AGAINST: {state['debate_plan'].get('against_position', '?')[:60]}")
    return state


def debate_researcher(state, tracker, side='for'):
    plan = state['debate_plan']
    topic = state['topic']
    if side == 'for':
        queries = [f"evidence supporting {plan['for_position']}",
                   f'benefits advantages {topic}', f'reasons why {topic}']
    else:
        queries = [f'evidence against {topic}',
                   f'problems disadvantages {topic}', f'criticism risks {topic}']
    for dim in plan.get('dimensions', [])[:2]:
        queries.append(f"{dim} {'benefits' if side == 'for' else 'problems'}")
    evidence = []
    for q in queries:
        evidence.extend(search(state['_index'], state['_all_chunks'], q, k=3))
    seen = set()
    unique = [c for c in evidence if c['index'] not in seen and not seen.add(c['index'])]
    state[f'evidence_{side}'] = unique[:8]
    log_agent(state, f'researcher_{side}', {'queries': len(queries)}, f'{len(unique[:8])} chunks')
    print(f'  [Researcher {side.upper()}] {len(unique[:8])} evidence chunks')
    return state


def debate_debater(state, tracker, side='for'):
    plan = state['debate_plan']
    evidence = state.get(f'evidence_{side}', [])
    evidence_text = '\n---\n'.join([e['text'] for e in evidence])
    position = plan['for_position'] if side == 'for' else plan['against_position']
    system = DEBATER_SYSTEM_TEMPLATE.format(side='FOR' if side == 'for' else 'AGAINST', position=position)
    result = call_llm_strong(system=system,
                             prompt=f"Topic: {state['topic']}\nDimensions: {', '.join(plan.get('dimensions', []))}\n\nEvidence:\n{evidence_text}",
                             json_output=True)
    tracker.record(result, f'debater_{side}')
    state[f'argument_{side}'] = parse_json(result['text'])
    log_agent(state, f'debater_{side}', {'evidence': len(evidence)},
              f"{len(state[f'argument_{side}'].get('arguments', []))} arguments",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Debater {side.upper()}] {len(state[f'argument_{side}'].get('arguments', []))} arguments")
    return state


def debate_cross_examiner(state, tracker, my_side='for'):
    opposing = 'against' if my_side == 'for' else 'for'
    opp_arg = state.get(f'argument_{opposing}', {})
    my_evidence = '\n---\n'.join([e['text'] for e in state.get(f'evidence_{my_side}', [])])
    result = call_llm_strong(
        system=CROSS_EXAM_SYS,
        prompt=f"You argue {my_side.upper()}.\nOpposing argument:\n{json.dumps(opp_arg, indent=2)}\n\nYour evidence:\n{my_evidence}",
        json_output=True)
    tracker.record(result, f'cross_exam_{my_side}')
    state[f'cross_exam_{my_side}'] = parse_json(result['text'])
    log_agent(state, f'cross_exam_{my_side}', {},
              state[f'cross_exam_{my_side}'].get('weakest_point', '?')[:50],
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print(f"  [Cross-Exam {my_side.upper()}] Challenged: {state[f'cross_exam_{my_side}'].get('weakest_point', '?')[:50]}")
    return state


def debate_judge(state, tracker, round_num=1):
    arg_for = state.get('argument_for', {})
    arg_against = state.get('argument_against', {})
    cross_ctx = ''
    if round_num > 1 and state.get('cross_exam_for'):
        cross_ctx = f"\n\nCROSS-EXAMINATION:\nFOR challenges: {json.dumps(state['cross_exam_for'])}\nAGAINST challenges: {json.dumps(state['cross_exam_against'])}\n\nFactor cross-exam into scoring."
    result = call_llm_strong(
        system=JUDGE_SYS,
        prompt=f"Topic: {state['topic']}\nRound: {round_num}\n\nFOR:\n{json.dumps(arg_for, indent=2)}\n\nAGAINST:\n{json.dumps(arg_against, indent=2)}{cross_ctx}",
        json_output=True)
    tracker.record(result, f'judge_r{round_num}')
    state['judgment'] = parse_json(result['text'])
    log_agent(state, f'judge_r{round_num}', {'round': round_num},
              f"Winner: {state['judgment'].get('winner', '?')} ({state['judgment'].get('margin', '?')})",
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    f_t = state['judgment'].get('for_score', {}).get('total', '?')
    a_t = state['judgment'].get('against_score', {}).get('total', '?')
    print(f"  [Judge R{round_num}] FOR: {f_t}/50 | AGAINST: {a_t}/50 | "
          f"Winner: {state['judgment'].get('winner', '?').upper()} ({state['judgment'].get('margin', '?')})")
    return state


def debate_synthesizer(state, tracker):
    j = state.get('judgment', {})
    af = state.get('argument_for', {})
    aa = state.get('argument_against', {})
    result = call_llm_strong(
        system=SYNTHESIZER_SYS,
        prompt=f"Topic: {state['topic']}\nVerdict: {j.get('winner', '?')} ({j.get('margin', '?')})\n"
               f"FOR thesis: {af.get('opening_statement', '?')}\nAGAINST thesis: {aa.get('opening_statement', '?')}\n"
               f"Strongest FOR: {j.get('strongest_point_for', '?')}\nStrongest AGAINST: {j.get('strongest_point_against', '?')}",
        json_output=True)
    tracker.record(result, 'synthesizer')
    state['synthesis'] = parse_json(result['text'])
    log_agent(state, 'synthesizer', {}, state['synthesis'].get('nuanced_conclusion', '')[:80],
              {'tokens': result['tokens'], 'latency_ms': result['latency_ms']})
    print('  [Synthesizer] Done')
    return state

print('Track B agents loaded!')

## Milestone 1: Wire the Sequential Debate Pipeline

**Your task:** Call agents in order: Planner -> Researchers -> Debaters -> Judge -> Synthesizer

**Important state setup:**
- `state['topic']` = debate topic
- `state['_index']` = FAISS index (used by researcher)
- `state['_all_chunks']` = raw chunks (used by researcher)
- `state['chunks']` = initial context chunks from search

In [ ]:
def run_debate_pipeline(pdf_path, topic, budget=1.00, max_rounds=2):
    """Run the full debate pipeline."""
    tracker = CostTracker(budget=budget)
    print(f"\n{'=' * 65}\n  MULTI-AGENT DEBATE SYSTEM\n{'=' * 65}")
    print(f'  Topic: {topic}\n  Budget: ${budget:.2f}\n  Max rounds: {max_rounds}')

    # Step 1: Load + index + init state
    print('\n[1] Loading document...')
    # YOUR CODE HERE
    # HINT:
    #   chunks = load_and_chunk(pdf_path)
    #   index = build_index(chunks)
    #   state = init_state(topic)
    #   state['topic'] = topic
    #   state['_index'] = index
    #   state['_all_chunks'] = chunks
    #   state['chunks'] = search(index, chunks, topic, k=8)


    # Step 2: Plan
    print('\n[2] Planning debate framework...')
    # YOUR CODE HERE
    # HINT: state = debate_planner(state, tracker)


    # Step 3: Research (run sequentially for now)
    print('\n[3] Gathering evidence...')
    # YOUR CODE HERE
    # HINT: state = debate_researcher(state, tracker, 'for')
    #       state = debate_researcher(state, tracker, 'against')


    # Step 4: Debate (run sequentially for now)
    print('\n[4] Building arguments...')
    # YOUR CODE HERE
    # HINT: state = debate_debater(state, tracker, 'for')
    #       state = debate_debater(state, tracker, 'against')


    # Step 5: Judge
    print('\n[5] Judging...')
    # YOUR CODE HERE
    # HINT: state = debate_judge(state, tracker, round_num=1)


    # Step 6: Synthesize
    print('\n[6] Synthesizing...')
    # YOUR CODE HERE
    # HINT: state = debate_synthesizer(state, tracker)


    # Print results
    j = state.get('judgment', {})
    s = state.get('synthesis', {})
    print(f"\n{'=' * 65}")
    print(f"  WINNER: {j.get('winner', '?').upper()} ({j.get('margin', '?')})")
    print(f"  FOR: {j.get('for_score', {}).get('total', '?')}/50 | AGAINST: {j.get('against_score', {}).get('total', '?')}/50")
    print(f"  Conclusion: {s.get('nuanced_conclusion', '?')}")
    print(f"{'=' * 65}")
    tracker.report()
    return state

# Uncomment when ready:
# DEBATE_TOPIC = 'Is supervised learning better than unsupervised learning for practical applications?'
# debate_state = run_debate_pipeline(PDF_PATH, DEBATE_TOPIC)

## Milestone 2: Parallel Researchers + Debaters

**Your task:** Run researchers in parallel, then debaters in parallel.

**Hints:**
- Researchers are independent (FOR and AGAINST don't depend on each other)
- Debaters are independent too
- Use same `ThreadPoolExecutor` pattern as Track A
- Key difference: these agents take a `side` parameter

In [ ]:
def run_parallel_debate(fns_with_args, state, tracker):
    """Run [(fn, side), ...] pairs in parallel."""
    # YOUR CODE HERE
    # HINT:
    #   with ThreadPoolExecutor(max_workers=len(fns_with_args)) as executor:
    #       futures = {}
    #       for fn, side in fns_with_args:
    #           s = {**state}  # Each gets own copy
    #           futures[executor.submit(fn, s, tracker, side)] = (fn.__name__, side)
    #       for future in as_completed(futures):
    #           name, side = futures[future]
    #           try:
    #               result_state = future.result()
    #               # Merge side-specific keys
    #               for key in [f'evidence_{side}', f'argument_{side}', f'cross_exam_{side}']:
    #                   if key in result_state:
    #                       state[key] = result_state[key]
    #               state['log'].extend(result_state.get('log', []))
    #           except Exception as e:
    #               print(f'  [ERROR] {name}_{side}: {e}')
    return state

# Usage in pipeline:
# state = run_parallel_debate([(debate_researcher, 'for'), (debate_researcher, 'against')], state, tracker)
# state = run_parallel_debate([(debate_debater, 'for'), (debate_debater, 'against')], state, tracker)

## Milestone 3: Cross-Examination Round

**Your task:** If the judge's margin is "razor-thin" or "narrow", trigger cross-examination.

**Requirements:**
1. After Judge Round 1, check `state['judgment']['margin']`
2. If close: run both cross-examiners in parallel, then Judge Round 2
3. Check budget before triggering (cross-exam costs ~$0.08-0.10)
4. Track `state['rounds_played']`

In [ ]:
# YOUR CODE HERE
# Add this logic after Step 5 (Judge R1) in run_debate_pipeline:
#
# margin = state.get('judgment', {}).get('margin', 'decisive')
# if max_rounds > 1 and margin in ('razor-thin', 'narrow') and tracker.remaining() > 0.15:
#     print(f'\n  Debate is {margin} - triggering cross-examination...')
#     state = run_parallel_debate(
#         [(debate_cross_examiner, 'for'), (debate_cross_examiner, 'against')],
#         state, tracker)
#     state = debate_judge(state, tracker, round_num=2)
#     state['rounds_played'] = 2
# else:
#     state['rounds_played'] = 1
#     print(f'  Skipping cross-examination: result was {margin}')

---
## Try Your Own

Upload your own PDF and run either pipeline:

In [ ]:
# Upload your own PDF
from google.colab import files
uploaded = files.upload()
YOUR_PDF = list(uploaded.keys())[0]
print(f'Uploaded: {YOUR_PDF}')

In [ ]:
# Run Intelligence Hub on your PDF
# hub_state = run_hub_pipeline(YOUR_PDF, budget=1.00, max_retries=1)

In [ ]:
# Run Debate on your PDF
# YOUR_TOPIC = 'Your debate topic here'
# debate_state = run_debate_pipeline(YOUR_PDF, YOUR_TOPIC, budget=1.00, max_rounds=2)